## Анализ данных

In [1]:
!pip install geopandas

In [2]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point

data_dir = "data"
results_dir = "analysis_results"
os.makedirs(results_dir, exist_ok=True)

def read_file(filename):
    return os.path.join(data_dir, filename)

def save(df, filename):
    path = os.path.join(results_dir, filename)
    df.to_csv(path, index=False)
    print(f"Cохранено: {path}")

### Анализ застройки

In [4]:
buildings = gpd.read_file(read_file("baikalsk_analiz_zastroyki.geojson"))

# Количество зданий по типам
by_type = (
    buildings.groupby("building")
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
print("Количество зданий по типам:")
print(by_type.to_string(index=False))
save(by_type, "buildings_by_type.csv")

# Этажность
floors_stats = (
    buildings.groupby("building")["floors"]
    .agg(["mean", "median", "min", "max"])
    .round(2)
    .reset_index()
)
floors_stats.columns = ["building", "mean_floors", "median_floors", "min_floors", "max_floors"]
print("\nСтатистика этажности по типам:")
print(floors_stats.to_string(index=False))
save(floors_stats, "buildings_floors_stats.csv")

# Площадь по типам
area_stats = (
    buildings.groupby("building")["area_m2"]
    .agg(
        total_area=("sum"),
        mean_area=("mean"),
        median_area=("median")
    )
    .round(1)
    .reset_index()
)
print("\nПлощадь зданий по типам (м^2):")
print(area_stats.to_string(index=False))
save(area_stats, "buildings_area_stats.csv")

# Общая статистика
print(f"\nВсего зданий: {len(buildings)}")
print(f"Суммарная площадь: {buildings['area_m2'].sum():,.0f} м^2")
print(f"Средняя площадь: {buildings['area_m2'].mean():,.1f} м^2")
print(f"Средняя этажность: {buildings['floors'].mean():.2f}")

Количество зданий по типам:
     building  count
   apartments    426
  residential    393
      garages     64
       public     27
        house     19
   commercial      7
       retail      4
   industrial      2
     hospital      1
        kiosk      1
         roof      1
train_station      1
Cохранено: analysis_results/buildings_by_type.csv

Статистика этажности по типам:
     building  mean_floors  median_floors  min_floors  max_floors
   apartments         3.01            3.0         1.0         5.0
   commercial         1.00            1.0         1.0         1.0
      garages         1.00            1.0         1.0         1.0
     hospital         1.00            1.0         1.0         1.0
        house         1.21            1.0         1.0         2.0
   industrial         1.00            1.0         1.0         1.0
        kiosk         1.00            1.0         1.0         1.0
       public         1.00            1.0         1.0         1.0
  residential         1

### Функциональное наполнение

In [5]:
poi_files = {
    "cafe": ("cafe_baikalsk_clean.csv", "csv"),
    "hotels": ("hotels_baikalsk.geojson", "geojson"),
    "parks": ("parks_baikalsk.geojson", "geojson"),
    "hospitals": ("hospitals_baikalsk.geojson","geojson"),
    "places": ("places_baikalsk.geojson", "geojson"),
}

labels = {
    "cafe": "Кафе и рестораны",
    "hotels": "Средства размещения",
    "parks": "Парки и скверы",
    "hospitals": "Медицинские учреждения",
    "places": "Достопримечательности",
}

poi_dfs = {}
for key, (filename, fmt) in poi_files.items():
    path = read_file(filename)
    if fmt == "csv":
        df = pd.read_csv(path)
    else:
        df = gpd.read_file(path)
    poi_dfs[key] = df

# Количество по категориям
poi_counts = pd.DataFrame([
    {"category": labels[k], "count": len(v)}
    for k, v in poi_dfs.items()
])
print("Количество POI по категориям:")
print(poi_counts.to_string(index=False))
print(f"Итого: {poi_counts['count'].sum()}")
save(poi_counts, "poi_by_category.csv")

# Рейтинги
print("\nСредний рейтинг по категориям:")
for key, df in poi_dfs.items():
    if "rating" in df.columns:
        ratings = pd.to_numeric(df["rating"], errors="coerce").dropna()
        if len(ratings) > 0:
            print(f"{labels[key]}: {ratings.mean():.2f} "
                  f"(n={len(ratings)}, min={ratings.min():.1f}, "
                  f"max={ratings.max():.1f})")

# Топ-10 по рейтингу
all_poi = []
for key, df in poi_dfs.items():
    tmp = df.copy()
    tmp["category"] = labels[key]
    if "rating" in tmp.columns:
        all_poi.append(tmp[["name", "category", "rating"]])

if all_poi:
    all_poi_df = pd.concat(all_poi, ignore_index=True)
    all_poi_df["rating"] = pd.to_numeric(all_poi_df["rating"], errors="coerce")
    top10 = (
        all_poi_df.dropna(subset=["rating"])
        .nlargest(10, "rating")[["name", "category", "rating"]]
        .reset_index(drop=True)
    )
    print("\nТоп-10 объектов по рейтингу:")
    print(top10.to_string(index=False))
    save(top10, "poi_top10_rating.csv")

# POI по районам
neighborhoods = gpd.read_file(read_file("baikalsk_neighborhoods.geojson"))
name_col = next(
    (c for c in ["district_name", "name"] if c in neighborhoods.columns),
    neighborhoods.columns[0]
)

poi_by_district = []
for key, df in poi_dfs.items():
    if "lat" not in df.columns or "lon" not in df.columns:
        continue
    df_clean = df.dropna(subset=["lat", "lon"]).copy()
    for _, nbr in neighborhoods.iterrows():
        cnt = sum(
            nbr.geometry.contains(Point(row.lon, row.lat))
            for _, row in df_clean.iterrows()
        )
        poi_by_district.append({
            "district": str(nbr[name_col]),
            "category": labels[key],
            "count":    cnt
        })

if poi_by_district:
    district_df = pd.DataFrame(poi_by_district)
    pivot = district_df.pivot_table(
        index="district", columns="category",
        values="count", aggfunc="sum", fill_value=0
    ).reset_index()
    pivot["total"] = pivot.iloc[:, 1:].sum(axis=1)
    pivot = pivot.sort_values("total", ascending=False)
    print("\nPOI по районам:")
    print(pivot.to_string(index=False))
    save(pivot, "poi_by_district.csv")

    # Разнообразие по районам
    diversity = (
        district_df[district_df["count"] > 0]
        .groupby("district")["category"]
        .nunique()
        .reset_index(name="diversity")
        .sort_values("diversity", ascending=False)
    )
    print("\nРазнообразие категорий по районам:")
    print(diversity.to_string(index=False))
    save(diversity, "poi_diversity_by_district.csv")

Количество POI по категориям:
              category  count
      Кафе и рестораны     28
   Средства размещения     32
        Парки и скверы      7
Медицинские учреждения      6
 Достопримечательности     66
Итого: 139
Cохранено: analysis_results/poi_by_category.csv

Средний рейтинг по категориям:
Средства размещения: 4.72 (n=31, min=3.1, max=5.0)
Парки и скверы: 4.75 (n=6, min=3.9, max=5.0)
Достопримечательности: 4.80 (n=28, min=3.0, max=5.0)

Топ-10 объектов по рейтингу:
                          name              category  rating
                Дом на Байкале   Средства размещения     5.0
          Байкальские рассветы   Средства размещения     5.0
               Байкальский дом   Средства размещения     5.0
                          Кедр   Средства размещения     5.0
            Хижина Дяди Сережи   Средства размещения     5.0
                   Green house   Средства размещения     5.0
                    Апарт-бюро   Средства размещения     5.0
              Колыбель Байкала  

/tmp/ipykernel_12019/2591262873.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_poi_df = pd.concat(all_poi, ignore_index=True)



POI по районам:
            district  Достопримечательности  Кафе и рестораны  Медицинские учреждения  Парки и скверы  Средства размещения  total
 микрорайон Гагарина                     21                 1                       3               2                    1     28
    микрорайон Южный                     18                 2                       0               1                    1     22
        Красный Ключ                      2                 2                       3               0                   11     18
микрорайон Строитель                      5                 0                       0               3                   10     18
Cохранено: analysis_results/poi_by_district.csv

Разнообразие категорий по районам:
            district  diversity
 микрорайон Гагарина          5
        Красный Ключ          4
    микрорайон Южный          4
микрорайон Строитель          3
Cохранено: analysis_results/poi_diversity_by_district.csv


### Транспортная доступность

In [13]:
stops = pd.read_csv(read_file("baikalsk_stops_routes_coords_final.csv"))
stops_unique = stops.drop_duplicates("stop_name").dropna(subset=["lat", "lon"])

print(f"\nВсего остановок: {len(stops_unique)}")

# Маршруты по остановкам
routes_per_stop = (
    stops.groupby("stop_name")["route"]
    .apply(lambda x: list(x.unique()))
    .reset_index()
)
routes_per_stop["route_count"] = routes_per_stop["route"].apply(len)
print("\nМаршруты по остановкам:")
print(routes_per_stop[["stop_name", "route_count", "route"]].to_string(index=False))
save(routes_per_stop[["stop_name", "route_count"]], "stops_route_count.csv")

# Остановки по районам
stops_by_district = []
for _, nbr in neighborhoods.iterrows():
    cnt = sum(
        nbr.geometry.contains(Point(row.lon, row.lat))
        for _, row in stops_unique.iterrows()
    )
    stops_by_district.append({
        "district": str(nbr[name_col]),
        "stop_count": cnt
    })

stops_district_df = pd.DataFrame(stops_by_district).sort_values(
    "stop_count", ascending=False
)
print("\nОстановки по районам:")
print(stops_district_df.to_string(index=False))
save(stops_district_df, "stops_by_district.csv")

# POI в буфере 500м для каждой остановки
buffer_m = 500
CRS_meter = "EPSG:32648"

all_poi_geo = []
for key, df in poi_dfs.items():
    if "lat" not in df.columns or "lon" not in df.columns:
        continue
    df_clean = df.dropna(subset=["lat", "lon"]).copy()

    if "category" in df_clean.columns:
        df_clean["category"] = df_clean["category"]
    elif "functional_type" in df_clean.columns:
        df_clean["category"] = df_clean["functional_type"]
    else:
        df_clean["category"] = key

    gdf_poi = gpd.GeoDataFrame(
        df_clean,
        geometry=gpd.points_from_xy(df_clean["lon"], df_clean["lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_meter)
    gdf_poi["CRS_meter"] = labels[key]
    all_poi_geo.append(gdf_poi[["category", "geometry"]])

if all_poi_geo:
    all_poi_gdf = pd.concat(all_poi_geo, ignore_index=True)

    stops_gdf = gpd.GeoDataFrame(
        stops_unique,
        geometry=gpd.points_from_xy(stops_unique["lon"], stops_unique["lat"]),
        crs="EPSG:4326"
    ).to_crs(CRS_meter)

    buffer_stats = []
    for _, stop in stops_gdf.iterrows():
        buffer = stop.geometry.buffer(buffer_m)
        in_buf = all_poi_gdf[all_poi_gdf.geometry.within(buffer)]
        buffer_stats.append({
            "stop_name": stop["stop_name"],
            "poi_total": len(in_buf),
            "categories": in_buf["category"].nunique(),
        })

    buffer_df = pd.DataFrame(buffer_stats).sort_values(
        "poi_total", ascending=False
    )
    print(f"\nPOI в буфере {buffer_m}м по остановкам:")
    print(buffer_df.to_string(index=False))
    save(buffer_df, "poi_in_buffer.csv")

    # Доля территории покрытой буферами
    boundary = gpd.read_file(read_file("baikalsk_boundary.geojson")).to_crs(CRS_meter)
    city_area = boundary.geometry.iloc[0].area
    union_buf = stops_gdf.geometry.buffer(buffer_m).unary_union
    covered = union_buf.intersection(boundary.geometry.iloc[0]).area
    coverage_pct = covered / city_area * 100

    print(f"\nПлощадь города: {city_area/1e6:.2f} км^2")
    print(f"Покрыто буферами: {covered/1e6:.2f} км^2")
    print(f"Доля покрытия: {coverage_pct:.1f}%")

    save(pd.DataFrame([{
        "city_area_km2": round(city_area/1e6, 2),
        "covered_km2": round(covered/1e6, 2),
        "coverage_pct": round(coverage_pct, 1),
    }]), "transport_coverage.csv")


Всего остановок: 15

Маршруты по остановкам:
        stop_name  route_count     route
           202 км            1       [4]
      Автостанция            3 [5, 6, 4]
         Гагарина            3 [5, 6, 4]
             Дачи            1       [4]
       Ж/д вокзал            1       [4]
              МЖК            3 [5, 6, 4]
             Маяк            3 [5, 6, 4]
        Общежития            3 [5, 6, 4]
Площадь Юбилейная            3 [5, 6, 4]
       Пост ГИБДД            2    [5, 6]
    Привокзальная            3 [5, 6, 4]
           Ракета            1       [4]
       Сангородок            2    [5, 6]
           Солзан            1       [4]
    Спорткомплекс            3 [5, 6, 4]
Cохранено: analysis_results/stops_route_count.csv

Остановки по районам:
            district  stop_count
микрорайон Строитель           4
 микрорайон Гагарина           1
    микрорайон Южный           1
        Красный Ключ           1
Cохранено: analysis_results/stops_by_district.csv

POI в буф

/tmp/ipykernel_12019/3713145830.py:90: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  union_buf = stops_gdf.geometry.buffer(buffer_m).unary_union


### Индекс NDVI

In [19]:
for city, filename in [
    ("Байкальск", "ndvi_stats_baikalsk.csv"),
    ("Иркутск", "ndvi_stats_irkutsk.csv"),
]:
    path = read_file(filename)

    df = pd.read_csv(path)
    df.columns = [c.strip().lower().replace("ndvi_", "") for c in df.columns]

    best_year = int(df.loc[df["mean"].idxmax(), "year"])
    worst_year = int(df.loc[df["mean"].idxmin(), "year"])
    trend = df["mean"].iloc[-1] - df["mean"].iloc[0]

    print(f"\n{city}:")
    print(f"Среднее NDVI за период: {df['mean'].mean():.3f}")
    print(f"Медиана NDVI за период: {df['median'].mean():.3f}")
    print(f"Лучший год: {best_year} "
          f"({df.loc[df['year']==best_year, 'mean'].iloc[0]:.3f})")
    print(f"Худший год: {worst_year} "
          f"({df.loc[df['year']==worst_year, 'mean'].iloc[0]:.3f})")
    print(f"Динамика 2017→2025:"
          f"{'увеличилась на' if trend > 0 else ' уменьшилась на'} {abs(trend):.3f}")

    print(f"\nNDVI по годам ({city}):")
    print(df[["year", "mean", "median", "stddev"]].round(3).to_string(index=False))

    save(df, f"ndvi_stats_{city.lower()}.csv")

# Сравнение городов
b = pd.read_csv(read_file("ndvi_stats_baikalsk.csv"))
i = pd.read_csv(read_file("ndvi_stats_irkutsk.csv"))
b.columns = [c.strip().lower().replace("ndvi_", "") for c in b.columns]
i.columns = [c.strip().lower().replace("ndvi_", "") for c in i.columns]

comparison = pd.DataFrame({
    "year": b["year"],
    "baikalsk_mean": b["mean"].round(3),
    "irkutsk_mean": i["mean"].round(3),
    "difference": (b["mean"] - i["mean"]).round(3),
})
print("\nСравнение городов по среднему NDVI:")
print(comparison.to_string(index=False))
save(comparison, "ndvi_comparison.csv")

print(f"\nБайкальск зеленее Иркутска в среднем на "
      f"{comparison['difference'].mean():.3f} единиц NDVI")


Байкальск:
Среднее NDVI за период: 0.635
Медиана NDVI за период: 0.708
Лучший год: 2018 (0.692)
Худший год: 2019 (0.590)
Динамика 2017→2025: уменьшилась на 0.063

NDVI по годам (Байкальск):
 year  mean  median  stddev
 2017 0.690   0.777   0.223
 2018 0.692   0.785   0.225
 2019 0.590   0.652   0.198
 2020 0.638   0.707   0.226
 2021 0.636   0.707   0.224
 2022 0.636   0.707   0.238
 2023 0.616   0.676   0.237
 2024 0.595   0.668   0.222
 2025 0.627   0.691   0.244
Cохранено: analysis_results/ndvi_stats_байкальск.csv

Иркутск:
Среднее NDVI за период: 0.467
Медиана NDVI за период: 0.542
Лучший год: 2017 (0.495)
Худший год: 2019 (0.440)
Динамика 2017→2025: уменьшилась на 0.023

NDVI по годам (Иркутск):
 year  mean  median  stddev
 2017 0.495   0.590   0.322
 2018 0.455   0.535   0.302
 2019 0.440   0.504   0.277
 2020 0.460   0.520   0.286
 2021 0.482   0.566   0.310
 2022 0.461   0.520   0.294
 2023 0.465   0.551   0.294
 2024 0.473   0.566   0.308
 2025 0.471   0.527   0.299
Cохранено